# Custom ResNet for Malaria Cell Classification — David Shumbusho

**Model:** Custom ResNet trained from scratch (Keras Subclassing API, residual blocks after He et al., 2016).
**Task:** classify thin-smear red blood cell images as *Uninfected (0)* or *Parasitized (1)*.
**Target (held-out test set):** sensitivity ≥ 90% and specificity ≥ 90%.

Code lives in the team repo: `members/david_custom_resnet/model.py` (architecture) and
`members/david_custom_resnet/train.py` (experiment runner). All four members use the same
shared split manifests in `splits/` (70/15/15, stratified, seed 42).

**Rules followed in this notebook**
- Experiments are compared on the **validation** set only.
- The **test** set is used once, in Section 13, after the final run has been chosen.
- Every run is logged to TensorBoard under `logs/fit/<run_name>` on Google Drive, so it survives Colab disconnects.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. Environment verification

In [ ]:
import tensorflow as tf, sys
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus)
assert gpus, "No GPU found: Runtime -> Change runtime type -> T4 GPU"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Setup: Google Drive, repository and dataset

- **Google Drive** stores every run's TensorBoard logs, weights and results (`MyDrive/malaria_cnn/david_custom_resnet/`).
  This folder is what gets shared in the report appendix.
- **Repository** is cloned fresh into the Colab VM. Set `BRANCH` to the branch containing the ResNet code
  (`main` once the PR is merged).
- **Dataset** (`cell_images.zip`, ~350 MB) is downloaded from the shared Drive link and unzipped into `data/cell_images/`,
  exactly where the shared split CSVs expect it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/N-Maurice/malaria_diagnosis_cnn.git"
BRANCH = "main"   # use "david/feature-custom-resnet" if the PR is not merged yet
REPO_DIR = "/content/malaria_diagnosis_cnn"
DATASET_DRIVE_ID = "1rMZb3yA7DOtSDjKv3ZdlahkpAjZDutFq"   # public cell_images.zip

import os
if not os.path.isdir(REPO_DIR):
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch -q && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import os
if not os.path.isdir("data/cell_images/Parasitized") or len(os.listdir("data/cell_images/Parasitized")) < 13000:
    !gdown -q {DATASET_DRIVE_ID} -O /content/cell_images.zip
    !unzip -q -o /content/cell_images.zip -d data/
for c in ("Parasitized", "Uninfected"):
    print(c, sum(f.endswith(".png") for f in os.listdir(f"data/cell_images/{c}")))

## 3. Imports and configuration

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.config import PROJECT_ROOT, SPLIT_DIR
from src.eval_utils import evaluate_binary, plot_confusion_matrix, plot_roc_curve
from src.gradcam_utils import make_gradcam_heatmap, overlay_heatmap
from members.david_custom_resnet.model import build_model
from members.david_custom_resnet.train import (
    ExperimentConfig, run_experiment, load_trained_model, evaluate_on_test, load_split,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/malaria_cnn/david_custom_resnet")
OUTPUT_ROOT = DRIVE_ROOT / "runs"        # weights, histories, predictions, metrics per run
LOG_ROOT = DRIVE_ROOT / "logs" / "fit"   # TensorBoard event files per run
FIG_DIR = DRIVE_ROOT / "figures"
for p in (OUTPUT_ROOT, LOG_ROOT, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)
!pip freeze > {DRIVE_ROOT}/environment.txt
print(OUTPUT_ROOT, LOG_ROOT, sep="\n")

## 4. Split loading and verification
The shared manifests are reused unchanged; every listed file must exist.

In [ ]:
splits = {name: load_split(name) for name in ("train", "val", "test")}
for name, frame in splits.items():
    missing = (~frame.filepath.map(lambda p: (PROJECT_ROOT / p).is_file())).sum()
    print(f"{name:5s} {len(frame):6d} images | {frame.label.value_counts().to_dict()} | missing files: {missing}")
    assert missing == 0

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row, (label, name) in enumerate([(1, "Parasitized"), (0, "Uninfected")]):
    sample = splits["train"][splits["train"].label == label].sample(6, random_state=1)
    for ax, path in zip(axes[row], sample.filepath):
        img = Image.open(PROJECT_ROOT / path)
        ax.imshow(img); ax.set_title(f"{name}\n{img.size[0]}x{img.size[1]}", fontsize=8); ax.axis("off")
plt.suptitle("Training examples (original sizes vary; resized to 128x128 for the model)")
plt.tight_layout(); plt.savefig(FIG_DIR / "dataset_examples.png", dpi=200); plt.show()

## 5. Model definition

`CustomResNet` (in `model.py`) is a `tf.keras.Model` subclass built from `ResidualBlock` layer subclasses:

- **Stem:** 3×3 conv → BatchNorm → ReLU → 2×2 max-pool (128 → 64 px)
- **Residual stages:** each `ResidualBlock` = 3×3 conv → BN → ReLU → 3×3 conv → BN, **+ shortcut**, → ReLU.
  The first block of every later stage uses stride 2 and a 1×1 projection shortcut so shapes match.
- **Head:** global average pooling → dropout → Dense(1) logit → sigmoid = P(Parasitized)
- `use_skip=False` removes only the shortcut (a plain CNN with identical layers), used for the residual ablation.
- `gradcam_outputs(x)` returns the last conv feature map and the logit from one forward pass, for Grad-CAM.

In [ ]:
for kwargs in ({}, {"stage_filters": (32, 64, 128, 256), "blocks_per_stage": (2, 2, 2, 2)}):
    m = build_model(**kwargs)
    print(kwargs or "baseline", f"-> {m.count_params():,} parameters")
m.summary()

## 6–9. Experiments (TensorBoard tracked)

Each experiment changes **one idea** relative to the best previous run, so every improvement can be traced
to a logged run. Shared settings unless stated: 128×128 input, Adam, LR 1e-3, batch size 32, up to 20 epochs,
early stopping on validation loss (patience 5, best weights restored), binary cross-entropy.

| # | Run name | Change being tested |
|---|---|---|
| 01 | `custom_resnet_exp_01_baseline` | 3 stages (32-64-128), 1 block each, BatchNorm, no augmentation/regularisation |
| 02 | `custom_resnet_exp_02_deeper_4stage` | 4 stages (32-64-128-256), 2 blocks each (ResNet-18-style depth) |
| 03 | `custom_resnet_exp_03_no_skip_ablation` | Exp 02 **without** skip connections — does the residual path matter? |
| 04 | `custom_resnet_exp_04_augmentation` | Flips, 90° rotations, brightness/contrast jitter |
| 05 | `custom_resnet_exp_05_dropout_l2` | Dropout 0.3 + L2 1e-4 |
| 06 | `custom_resnet_exp_06_lr_3e-4` | Lower learning rate |
| 07 | `custom_resnet_exp_07_sgd_momentum` | SGD + Nesterov momentum vs Adam |
| 08 | `custom_resnet_exp_08_plateau_schedule` | Reduce-LR-on-plateau, 30 epochs (final candidate) |

**This plan is a starting point, not a script to run blindly.** After each run, look at the summary table and
TensorBoard, write down what happened, and decide what the next run should build on (the `base` for the
next cell). If a change hurts, the next experiment derives from the earlier, better config — that decision is
what the report and the oral defence will ask about.

Re-running a finished cell is safe: completed runs are loaded from Drive, not retrained.

### TensorBoard
Run once; click the refresh icon inside TensorBoard after each experiment.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_ROOT}

In [ ]:
def show_summary():
    table = pd.read_csv(OUTPUT_ROOT / "experiments_summary.csv")
    cols = ["run_name", "val_accuracy", "val_precision", "val_recall", "val_specificity", "val_f1",
            "val_roc_auc", "best_epoch", "epochs_completed", "train_accuracy_at_best", "val_accuracy_at_best",
            "train_minutes"]
    return table[cols].drop_duplicates("run_name", keep="last").set_index("run_name")

### Experiment 01 — baseline

In [ ]:
exp01 = ExperimentConfig(
    1, "baseline",
    hypothesis="A small 3-stage ResNet with BatchNorm establishes the reference performance.",
)
run_experiment(exp01, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 01):** _write what you see — train vs val curves, sensitivity vs specificity, signs of over/underfitting._

### Experiment 02 — deeper network

In [ ]:
exp02 = exp01.next(
    2, "deeper_4stage",
    hypothesis="More residual blocks and a 4th stage add capacity and a larger receptive field.",
    stage_filters=(32, 64, 128, 256), blocks_per_stage=(2, 2, 2, 2),
)
run_experiment(exp02, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 02):** _did depth help, hurt, or make no difference? Which architecture do later runs build on?_

### Experiment 03 — skip-connection ablation

In [ ]:
exp03 = exp02.next(
    3, "no_skip_ablation",
    hypothesis="Removing shortcuts from the deeper net slows optimisation / lowers accuracy, isolating the residual effect.",
    use_skip=False,
)
run_experiment(exp03, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 03):** _compare the loss curves of 02 vs 03 in TensorBoard. This ablation is not carried forward._

### Experiment 04 — data augmentation
Set `base` to the better of Exp 01 / Exp 02.

In [ ]:
base = exp02   # <- change to exp01 if the baseline architecture was better
exp04 = base.next(
    4, "augmentation",
    hypothesis="Cells have no fixed orientation and staining varies; flips/rotations/colour jitter should reduce overfitting.",
    augmentation="flip_rot_color",
)
run_experiment(exp04, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 04):**

### Experiment 05 — dropout + L2 regularisation

In [ ]:
base = exp04   # <- keep augmentation only if it helped
exp05 = base.next(
    5, "dropout_l2",
    hypothesis="Dropout before the classifier and L2 weight decay further narrow the train/validation gap.",
    dropout=0.3, l2=1e-4,
)
run_experiment(exp05, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 05):**

### Experiment 06 — learning rate

In [ ]:
base = exp05   # <- best run so far
exp06 = base.next(
    6, "lr_3e-4",
    hypothesis="A smaller step size gives smoother validation curves and a better minimum.",
    learning_rate=3e-4,
)
run_experiment(exp06, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 06):**

### Experiment 07 — optimizer

In [ ]:
base = exp06   # <- best run so far
exp07 = base.next(
    7, "sgd_momentum",
    hypothesis="SGD with Nesterov momentum (the optimiser used by He et al.) may generalise better than Adam.",
    optimizer="sgd", learning_rate=1e-2,
)
run_experiment(exp07, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 07):**

### Experiment 08 — learning-rate schedule (final candidate)

In [ ]:
base = exp06   # <- best of exp06 / exp07
exp08 = base.next(
    8, "plateau_schedule",
    hypothesis="Halving the LR when validation loss plateaus, with more epochs, lets the best config converge further.",
    lr_schedule="reduce_on_plateau", epochs=30, early_stopping_patience=7,
)
run_experiment(exp08, OUTPUT_ROOT, LOG_ROOT)
show_summary()

**Observations (Exp 08):**

_Optional extra runs if needed: `use_batchnorm=False`, `batch_size=64`, `lr_schedule="cosine"`. Use numbers 09, 10, …_

## 10. Experiment comparison (TensorBoard export)
Validation curves for every run, saved at high resolution for the report appendix.

In [ ]:
summary = show_summary()
display(summary)
summary.to_csv(FIG_DIR / "custom_resnet_experiments_table.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for run in summary.index:
    h = pd.read_csv(OUTPUT_ROOT / run / "history.csv")
    label = run.replace("custom_resnet_", "")
    axes[0].plot(h.epoch + 1, h.val_accuracy, label=label)
    axes[1].plot(h.epoch + 1, h.val_loss, label=label)
axes[0].set(title="Validation accuracy by experiment", xlabel="Epoch", ylabel="Accuracy")
axes[1].set(title="Validation loss by experiment", xlabel="Epoch", ylabel="Binary cross-entropy")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "experiment_comparison.png", dpi=300); plt.show()

## 11. Final experiment selection

Choose the run with the best **validation** evidence (sensitivity and specificity both high, highest ROC-AUC,
healthy train/val gap) — not just the top accuracy. Write the justification below.

In [ ]:
FINAL_RUN = ""   # e.g. "custom_resnet_exp_08_plateau_schedule"
assert FINAL_RUN in summary.index, "Set FINAL_RUN to one of the run names above"
display(summary.loc[[FINAL_RUN]])

**Why this run:** _your justification._

## 12. Training curves of the final model

In [ ]:
h = pd.read_csv(OUTPUT_ROOT / FINAL_RUN / "history.csv")
best = int(h.val_loss.idxmin())
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
    ax.plot(h.epoch + 1, h[metric], "o-", label=f"Training {metric}")
    ax.plot(h.epoch + 1, h[f"val_{metric}"], "o-", label=f"Validation {metric}")
    ax.axvline(best + 1, color="gray", ls="--", label=f"Restored best epoch ({best + 1})")
    ax.set(title=f"{title} — {FINAL_RUN}", xlabel="Epoch", ylabel=title); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "final_accuracy_loss_curves.png", dpi=300); plt.show()

**Interpretation:** _over/underfitting? where do the curves diverge? what did early stopping do?_

## 13. Held-out test evaluation (run once)

Everything above used validation data only. This cell evaluates the frozen final model on the test set,
saves the predictions and logs `test/*` scalars to the run's TensorBoard folder.

In [ ]:
test_metrics, test_preds = evaluate_on_test(FINAL_RUN, OUTPUT_ROOT, LOG_ROOT, threshold=0.5)
y_true, y_prob = test_preds.label.to_numpy(), test_preds.probability.to_numpy()
report = pd.Series(test_metrics).rename("test").to_frame()
report.index = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Sensitivity (recall, Parasitized)",
                "Specificity (TNR, Uninfected)"]
display(report.round(4))
print("Target sensitivity >= 0.90:", test_metrics["sensitivity"] >= 0.90)
print("Target specificity >= 0.90:", test_metrics["specificity"] >= 0.90)

In [ ]:
fig_cm, _ = plot_confusion_matrix(y_true, y_prob, output_path=FIG_DIR / "final_confusion_matrix.png")
fig_roc, _, auc = plot_roc_curve(y_true, y_prob, output_path=FIG_DIR / "final_roc_curve.png")
plt.show()

In [ ]:
# The four required plots in one row, as the report layout asks.
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))
for ax, metric in zip(axes[:2], ("accuracy", "loss")):
    ax.plot(h.epoch + 1, h[metric], "o-", label="Training"); ax.plot(h.epoch + 1, h[f"val_{metric}"], "o-", label="Validation")
    ax.set(title=metric.capitalize(), xlabel="Epoch"); ax.legend()
ConfusionMatrixDisplay.from_predictions(y_true, y_prob >= 0.5, display_labels=["Uninfected", "Parasitized"],
                                        cmap="Blues", ax=axes[2], colorbar=False)
axes[2].set_title("Confusion matrix (test)")
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[3].plot(fpr, tpr, label=f"AUC = {test_metrics['roc_auc']:.3f}"); axes[3].plot([0, 1], [0, 1], "--", color="gray")
axes[3].set(title="ROC curve (test)", xlabel="False-positive rate (1 - specificity)", ylabel="Sensitivity"); axes[3].legend()
plt.suptitle(f"Custom ResNet — {FINAL_RUN}"); plt.tight_layout()
plt.savefig(FIG_DIR / "final_four_plots_row.png", dpi=300); plt.show()

**Interpretation:** _confusion matrix — which error type dominates and why it matters clinically; ROC — what the AUC and curve shape say about the sensitivity/specificity trade-off._

## 14. Grad-CAM explainability

Grad-CAM uses the gradient of the Parasitized **logit** with respect to the last residual stage's feature map
(`gradcam_outputs`). Required examples: a correctly classified infected cell, a correctly classified uninfected
cell, and misclassified cells. For each, the heatmap shown is for the class the model predicted.

In [ ]:
model, final_cfg = load_trained_model(FINAL_RUN, OUTPUT_ROOT)
size = final_cfg.image_size
test_preds["predicted"] = (test_preds.probability >= 0.5).astype(int)
test_preds["correct"] = test_preds.predicted == test_preds.label

def gradcam_figure(rows, title, filename):
    fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3.1 * len(rows)), squeeze=False)
    for ax_row, (_, r) in zip(axes, rows.iterrows()):
        original = np.asarray(Image.open(PROJECT_ROOT / r.filepath).convert("RGB"))
        model_input = np.asarray(Image.fromarray(original).resize(size[::-1], Image.BILINEAR), dtype="float32") / 255.0
        heatmap = make_gradcam_heatmap(model, model_input, feature_model=model.gradcam_outputs,
                                       class_index=int(r.predicted), from_logits=True)
        names = ["Uninfected", "Parasitized"]
        ax_row[0].imshow(original); ax_row[0].set_title(f"True: {names[r.label]}\nP(parasitized)={r.probability:.3f}", fontsize=9)
        ax_row[1].imshow(heatmap, cmap="jet"); ax_row[1].set_title(f"Grad-CAM for '{names[r.predicted]}'", fontsize=9)
        ax_row[2].imshow(overlay_heatmap(original, heatmap)); ax_row[2].set_title(Path(r.filepath).name[:28], fontsize=7)
        for ax in ax_row: ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(FIG_DIR / filename, dpi=250); plt.show()

rng = 7
gradcam_figure(test_preds[test_preds.correct & (test_preds.label == 1)].sample(3, random_state=rng),
               "Correctly classified INFECTED cells", "gradcam_correct_infected.png")
gradcam_figure(test_preds[test_preds.correct & (test_preds.label == 0)].sample(3, random_state=rng),
               "Correctly classified UNINFECTED cells", "gradcam_correct_uninfected.png")
wrong = test_preds[~test_preds.correct].assign(confidence=lambda d: (d.probability - 0.5).abs())
if len(wrong):
    gradcam_figure(wrong.nlargest(min(4, len(wrong)), "confidence"),
                   "MISCLASSIFIED cells (most confident mistakes)", "gradcam_incorrect.png")

**Interpretation (answer in your own words):**
- Which regions drive each prediction? Is the attention on the cell or on the background/border?
- For infected cells, does the heat fall on the purple-stained parasite (ring/trophozoite/gametocyte)?
- What changes in the misclassified examples — diffuse attention, focus on artefacts or stain debris?
- Any sign of shortcuts (image borders, cell size, background colour)?
- Can Grad-CAM *prove* medically correct reasoning? Why / why not?

## 15. Error analysis

In [ ]:
tn, fp, fn, tp = pd.crosstab(test_preds.label, test_preds.predicted).reindex(index=[0, 1], columns=[0, 1], fill_value=0).to_numpy().ravel()
print(f"False negatives (missed infections): {fn} | False positives (false alarms): {fp} | of {len(test_preds)} test cells")

def gallery(frame, title, filename, n=8):
    frame = frame.head(n)
    fig, axes = plt.subplots(1, max(len(frame), 1), figsize=(2.2 * max(len(frame), 1), 2.8), squeeze=False)
    for ax, (_, r) in zip(axes[0], frame.iterrows()):
        ax.imshow(Image.open(PROJECT_ROOT / r.filepath)); ax.set_title(f"P(par)={r.probability:.2f}", fontsize=8); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(FIG_DIR / filename, dpi=200); plt.show()

gallery(test_preds[(test_preds.label == 1) & ~test_preds.correct].sort_values("probability"),
        "False negatives: Parasitized predicted Uninfected", "errors_false_negatives.png")
gallery(test_preds[(test_preds.label == 0) & ~test_preds.correct].sort_values("probability", ascending=False),
        "False positives: Uninfected predicted Parasitized", "errors_false_positives.png")
test_preds[~test_preds.correct].to_csv(FIG_DIR / "misclassified_test_cells.csv", index=False)

In [ ]:
# How confident is the model on its mistakes vs correct predictions?
plt.figure(figsize=(7, 4))
for label, name in ((0, "Uninfected"), (1, "Parasitized")):
    plt.hist(test_preds[test_preds.label == label].probability, bins=50, alpha=0.6, label=f"True {name}")
plt.axvline(0.5, color="k", ls="--", label="Threshold 0.5")
plt.yscale("log"); plt.xlabel("Predicted P(parasitized)"); plt.ylabel("Count (log)"); plt.legend()
plt.title("Test probability distribution"); plt.tight_layout(); plt.savefig(FIG_DIR / "probability_histogram.png", dpi=200); plt.show()

**Error analysis (your own words):**
- 2–3 representative misclassifications: true vs predicted label, probability, likely cause (e.g. faint/early ring
  stage, staining artefact or platelet overlapping the cell, possible label noise, blurry image).
- Overfitting/underfitting: which symptom appeared in which run (cite the curve), and what you changed in response.

## 16. Final experiment selection summary and artefacts

Everything is on Google Drive under `MyDrive/malaria_cnn/david_custom_resnet/`:
- `logs/fit/` — raw TensorBoard event logs for all runs (share this folder in the report appendix)
- `runs/<run_name>/` — config, history, predictions, metrics and weights per run
- `runs/experiments_summary.csv` and `figures/` — tables and high-resolution figures for the report

Copy the per-experiment rows into `members/david_custom_resnet/experiments_log.md` in the repo.